In [7]:
from glob import glob

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [9]:
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10")

class CifarData(torch.utils.data.Dataset):
  def __init__(self, split):
    self.data = cifar[split]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, i):
    item = self.data[i]
    return transform(item["img"].convert("RGB")), item["label"]

train_data = CifarData("train")
test_data = CifarData("test")

# num_workers=0: on Windows, worker processes can't load a Dataset class defined in a notebook
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0)

In [10]:
image, label = train_data[0]

In [11]:
image.size()

torch.Size([3, 32, 32])

In [12]:
class NeuralNet(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1=nn.Conv2d(3,12,5)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2=nn.Conv2d(12,24,5)
    self.fc1=nn.Linear(24*5*5,120)
    self.fc2=nn.Linear(120,84)
    self.fc3=nn.Linear(84,10)
  def forward(self,x):
    x= self.pool(F.relu(self.conv1(x)))
    x= self.pool(F.relu(self.conv2(x)))
    x = torch.flatten(x,1)
    x= F.relu(self.fc1(x))
    x= F.relu(self.fc2(x))
    x= self.fc3(x)
    return x

In [13]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer=optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [14]:
for epoch in range (30):
  print(f"Training Epoch {epoch}...")

  running_loss = 0.0

  for i, data in enumerate(train_loader):
    inputs, labels = data

    optimizer.zero_grad()
    output = net(inputs)

    loss = loss_function(output, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  print(f'Loss: {running_loss/len(train_loader):.4f}')

Training Epoch 0...
Loss: 2.1676
Training Epoch 1...
Loss: 1.7201
Training Epoch 2...
Loss: 1.5130
Training Epoch 3...
Loss: 1.3881
Training Epoch 4...
Loss: 1.3031
Training Epoch 5...
Loss: 1.2226
Training Epoch 6...
Loss: 1.1551
Training Epoch 7...
Loss: 1.0958
Training Epoch 8...
Loss: 1.0446
Training Epoch 9...
Loss: 1.0040
Training Epoch 10...
Loss: 0.9640
Training Epoch 11...
Loss: 0.9262
Training Epoch 12...
Loss: 0.8917
Training Epoch 13...
Loss: 0.8625
Training Epoch 14...
Loss: 0.8280
Training Epoch 15...
Loss: 0.7955
Training Epoch 16...
Loss: 0.7695
Training Epoch 17...
Loss: 0.7421
Training Epoch 18...
Loss: 0.7186
Training Epoch 19...
Loss: 0.6922
Training Epoch 20...
Loss: 0.6664
Training Epoch 21...
Loss: 0.6453
Training Epoch 22...
Loss: 0.6203
Training Epoch 23...
Loss: 0.5925
Training Epoch 24...
Loss: 0.5770
Training Epoch 25...
Loss: 0.5518
Training Epoch 26...
Loss: 0.5336
Training Epoch 27...
Loss: 0.5128
Training Epoch 28...
Loss: 0.4949
Training Epoch 29...
Los

In [15]:
torch.save(net.state_dict(), 'trained_net.pth')

In [16]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [17]:
correct = 0
total = 0

net.eval()
with torch.no_grad():
  for data in test_loader:
    images, labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 68 %


In [23]:
class_names=['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']

new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
  image = Image.open(image_path).convert("RGB")
  image = new_transform(image)
  image = image.unsqueeze(0)
  return image


image_paths = sorted(glob('test_images/*'))
images = [load_image(img) for img in image_paths]

net.eval()
with torch.no_grad():
  for path, image in zip(image_paths, images):
    output = net(image)
    _, predicted = torch.max(output, 1)
    print(f"{path} -> Prediction: {class_names[predicted.item()]}")


test_images\car.jpg -> Prediction: car
test_images\horse.jpg -> Prediction: horse
